<a href="https://colab.research.google.com/github/greek-nlp/benchmark/blob/main/gr_retrieval_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BarzokasDt Retrieval Analysis

This notebook prepares a retrieval-oriented view of `BarzokasDt`, where:
- **query** = `title`
- **passage** = `text`

It loads the dataset, shows sample titles, computes token-level statistics for `title` and `text`, and analyzes author distribution.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data_wrapper import BarzokasDt

In [ ]:
# Load dataset registry and BarzokasDt
gr_data = pd.read_csv('data.csv')
barzokas_dt = BarzokasDt(datasets=gr_data)

# Combine train + test for full-corpus retrieval analysis
barzokas_train = barzokas_dt.get('train')
barzokas_test = barzokas_dt.get('test')
barzokas_df = pd.concat([barzokas_train, barzokas_test], ignore_index=True).drop_duplicates().reset_index(drop=True)

print('Train shape:', barzokas_train.shape)
print('Test shape :', barzokas_test.shape)
print('Full shape :', barzokas_df.shape)
barzokas_df.head(3)

In [ ]:
# Keep only required columns for retrieval and analysis
retrieval_df = barzokas_df[['title', 'text', 'author']].copy()
retrieval_df['title'] = retrieval_df['title'].fillna('').astype(str).str.strip()
retrieval_df['text'] = retrieval_df['text'].fillna('').astype(str).str.strip()
retrieval_df['author'] = retrieval_df['author'].fillna('UNKNOWN').astype(str).str.strip()

retrieval_df.head()

In [ ]:
# Sample titles (query examples)
sample_n = min(20, len(retrieval_df))
title_samples = retrieval_df[['title', 'author']].sample(n=sample_n, random_state=42).reset_index(drop=True)
title_samples

In [ ]:
# Token-level statistics (simple whitespace tokenization)
def token_count(text: str) -> int:
    return len(text.split()) if isinstance(text, str) and text else 0

retrieval_df['title_tokens'] = retrieval_df['title'].apply(token_count)
retrieval_df['text_tokens'] = retrieval_df['text'].apply(token_count)

token_stats = pd.DataFrame({
    'title_tokens': retrieval_df['title_tokens'],
    'text_tokens': retrieval_df['text_tokens']
}).describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]).T

token_stats['total_tokens'] = [retrieval_df['title_tokens'].sum(), retrieval_df['text_tokens'].sum()]
token_stats

In [ ]:
# Number of authors + distribution
n_authors = retrieval_df['author'].nunique()
print('Unique authors:', n_authors)

author_dist = retrieval_df['author'].value_counts().rename_axis('author').reset_index(name='count')
author_dist['percentage'] = (author_dist['count'] / len(retrieval_df) * 100).round(2)
author_dist.head(30)

In [ ]:
# Visualize author distribution (top 25 authors)
top_k = 25
plot_df = author_dist.head(top_k).copy()

plt.figure(figsize=(10, 8))
sns.barplot(data=plot_df, y='author', x='count', color='#2a9d8f')
plt.title(f'Top {top_k} Authors by Number of Passages')
plt.xlabel('Count')
plt.ylabel('Author')
plt.tight_layout()
plt.show()

In [ ]:
# Optional: retrieval-ready dataframe preview
retrieval_ready = retrieval_df[['title', 'text', 'author', 'title_tokens', 'text_tokens']].copy()
retrieval_ready.sample(min(5, len(retrieval_ready)), random_state=42)